In [1]:
import pandas as pd

# Datasets info

In [2]:
import os

# Path to the datasets folder
datasets_folder = 'datasets'

# List all files in the datasets folder
files = os.listdir(datasets_folder)

# Initialize a report dictionary
report = {}

# Loop through each file and read the dataset
for file in files:
    file_path = os.path.join(datasets_folder, file)
    if file.endswith('.csv'):
        df_copy = pd.read_csv(file_path)
        if 'Class' in df_copy.columns:
            class_column = 'Class'
        elif 'class' in df_copy.columns:
            class_column = 'class'
        elif 'game' in df_copy.columns:
            class_column = 'game'
        elif 'V11' in df_copy.columns:
            class_column = 'V11'
        else:
            class_column = None

        if class_column:
            report[file] = {'rows': df_copy.shape[0], 'columns': df_copy.shape[1], 'classes': len(df_copy[class_column].unique())}
        else:
            report[file] = {'rows': df_copy.shape[0], 'columns': df_copy.shape[1], 'classes': 'N/A'}

# Remove entries with 'N/A' classes from the report
filtered_report = {k: v for k, v in report.items() if v['classes'] != 'N/A'}

# Sort the filtered report by number of classes, then by number of rows, and then by number of columns
sorted_report = dict(sorted(filtered_report.items(), key=lambda item: (item[1]['classes'], item[1]['rows'], item[1]['columns'])))

# Print the sorted report
for file, info in sorted_report.items():
    print(f"Dataset: {file}, Rows: {info['rows']}, Columns: {info['columns']}, Classes: {info['classes']}")

Dataset: Nursery.csv, Rows: 12960, Columns: 9, Classes: 5
Dataset: Phishing URL.csv, Rows: 18982, Columns: 80, Classes: 5
Dataset: Satimage.csv, Rows: 6430, Columns: 37, Classes: 6
Dataset: HAR.csv, Rows: 10299, Columns: 562, Classes: 6
Dataset: Mosquitoes.csv, Rows: 158249, Columns: 54, Classes: 6
Dataset: Dermatology.csv, Rows: 1000000, Columns: 35, Classes: 6
Dataset: Covertype.csv, Rows: 110393, Columns: 55, Classes: 7
Dataset: Land-use.csv, Rows: 9144, Columns: 221, Classes: 8
Dataset: Mfeat.csv, Rows: 2000, Columns: 7, Classes: 10
Dataset: Avila.csv, Rows: 20867, Columns: 11, Classes: 12
Dataset: Chess game.csv, Rows: 28056, Columns: 7, Classes: 18
Dataset: Walking.csv, Rows: 149332, Columns: 5, Classes: 22


# Nursery dataset

## Dataset analysis

In [3]:
dataset = "datasets/Nursery.csv"
def print_bad_lines(line):
    print(f"Bad line: {line}")

df = pd.read_csv(dataset, on_bad_lines=print_bad_lines, engine='python')
df

,parents,has_nurs,form,children,housing,finance,social,health,class
0,usual,proper,complete,1,convenient,convenient,nonprob,recommended,recommend
1,usual,proper,complete,1,convenient,convenient,nonprob,priority,priority
2,usual,proper,complete,1,convenient,convenient,nonprob,not_recom,not_recom
3,usual,proper,complete,1,convenient,convenient,slightly_prob,recommended,recommend
4,usual,proper,complete,1,convenient,convenient,slightly_prob,priority,priority
...,...,...,...,...,...,...,...,...,...
12955,great_pret,very_crit,foster,more,critical,inconv,slightly_prob,priority,spec_prior
12956,great_pret,very_crit,foster,more,critical,inconv,slightly_prob,not_recom,not_recom
12957,great_pret,very_crit,foster,more,critical,inconv,problematic,recommended,spec_prior
12958,great_pret,very_crit,foster,more,critical,inconv,problematic,priority,spec_prior


In [4]:
import plotly.express as px

fig = px.pie(df, names='class', title='Class Distribution', hole=0.3)
fig.update_traces(textinfo='percent+label')
fig.show()

## Preprocessing

In [5]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Define the categories with the desired order
categories = [
    ['usual', 'pretentious', 'great_pret'],  # parents
    ['proper', 'less_proper', 'improper', 'critical', 'very_crit'],  # has_nurs
    ['complete', 'completed', 'incomplete', 'foster'],  # form
    ['1', '2', '3', 'more'],  # children
    ['convenient', 'less_conv', 'critical'],  # housing
    ['convenient', 'inconv'],  # finance
    ['nonprob', 'slightly_prob', 'problematic'],  # social
    ['recommended', 'priority', 'not_recom']  # health
]

# Initialize the OrdinalEncoder with the specified categories
ordinal_encoder = OrdinalEncoder(categories=categories)

classes = df.pop('class')
columns = df.columns

# Fit and transform the data
df = ordinal_encoder.fit_transform(df)

# Convert the result back to a DataFrame for better readability
df = pd.DataFrame(df, columns=columns)
df['class'] = classes

df

,parents,has_nurs,form,children,housing,finance,social,health,class
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,recommend
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,priority
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,not_recom
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,recommend
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,priority
...,...,...,...,...,...,...,...,...,...
12955,2.0,4.0,3.0,3.0,2.0,1.0,1.0,1.0,spec_prior
12956,2.0,4.0,3.0,3.0,2.0,1.0,1.0,2.0,not_recom
12957,2.0,4.0,3.0,3.0,2.0,1.0,2.0,0.0,spec_prior
12958,2.0,4.0,3.0,3.0,2.0,1.0,2.0,1.0,spec_prior


## 1st labeling strategy (find worst pos/neg class then worst neg combination)

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pdb

def generate_prediction(df):
    # Split the data into features and target
    X = df.drop('class', axis=1)
    y = df['class']

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=40)

    # Create and train the Random Forest model
    rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
    rf_model.fit(X_train, y_train)

    # Make predictions with the Random Forest model
    rf_y_pred = rf_model.predict(X_test)
    # pdb.set_trace()

    # Evaluate the Random Forest model using AUC metric
    rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1])
    return rf_auc

In [22]:
from itertools import combinations

classes = ['recommend', 'priority', 'not_recom', 'very_recom', 'spec_prior']

# Function to generate all possible combinations of 2 groups with variable sizes
def generate_combinations(classes, half=True):
    all_combinations = []
    n = len(classes)

    if half: total = n // 2 + 1
    else: total = n
    
    for i in range(1, total):
        for group1 in combinations(classes, i):
            group2 = tuple(set(classes) - set(group1))
            if half & (len(group1) <= len(group2)):
                all_combinations.append((group1, group2))
            else:
                all_combinations.append((group1, group2))
    return all_combinations

# Generate and print all combinations
combinations_2_groups = generate_combinations(classes, half=True)

for combo in combinations_2_groups:
    print(combo)

# Print the total number of combinations
print(f'Total number of combinations: {len(combinations_2_groups)}')

(('recommend',), ('spec_prior', 'very_recom', 'priority', 'not_recom'))
(('priority',), ('spec_prior', 'recommend', 'very_recom', 'not_recom'))
(('not_recom',), ('spec_prior', 'recommend', 'very_recom', 'priority'))
(('very_recom',), ('spec_prior', 'recommend', 'priority', 'not_recom'))
(('spec_prior',), ('recommend', 'very_recom', 'priority', 'not_recom'))
(('recommend', 'priority'), ('spec_prior', 'very_recom', 'not_recom'))
(('recommend', 'not_recom'), ('spec_prior', 'very_recom', 'priority'))
(('recommend', 'very_recom'), ('spec_prior', 'priority', 'not_recom'))
(('recommend', 'spec_prior'), ('very_recom', 'priority', 'not_recom'))
(('priority', 'not_recom'), ('spec_prior', 'recommend', 'very_recom'))
(('priority', 'very_recom'), ('spec_prior', 'recommend', 'not_recom'))
(('priority', 'spec_prior'), ('recommend', 'very_recom', 'not_recom'))
(('not_recom', 'very_recom'), ('spec_prior', 'recommend', 'priority'))
(('not_recom', 'spec_prior'), ('recommend', 'very_recom', 'priority'))
(

In [23]:
result_df = pd.DataFrame(columns=['Positive', 'Negative', 'AUC'])

for group1, group2 in combinations_2_groups:
    df_copy = df.copy()
    df_copy['class'] = df_copy['class'].apply(lambda x: 'P' if x in group1 else 'N')

    auc = generate_prediction(df_copy)
    
    result = {'Positive': group1, 'Negative': group2, 'AUC': auc}
    result_df = pd.concat([result_df, pd.DataFrame([result])], ignore_index=True)
    print(result_df)
    

       Positive                                       Negative  AUC
0  (recommend,)  (spec_prior, very_recom, priority, not_recom)  1.0


C:\Users\joaop\AppData\Local\Temp\ipykernel_3576\3114288893.py:10: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



       Positive                                        Negative       AUC
0  (recommend,)   (spec_prior, very_recom, priority, not_recom)  1.000000
1   (priority,)  (spec_prior, recommend, very_recom, not_recom)  0.999875
       Positive                                        Negative       AUC
0  (recommend,)   (spec_prior, very_recom, priority, not_recom)  1.000000
1   (priority,)  (spec_prior, recommend, very_recom, not_recom)  0.999875
2  (not_recom,)   (spec_prior, recommend, very_recom, priority)  1.000000
        Positive                                        Negative       AUC
0   (recommend,)   (spec_prior, very_recom, priority, not_recom)  1.000000
1    (priority,)  (spec_prior, recommend, very_recom, not_recom)  0.999875
2   (not_recom,)   (spec_prior, recommend, very_recom, priority)  1.000000
3  (very_recom,)    (spec_prior, recommend, priority, not_recom)  0.999903
        Positive                                        Negative       AUC
0   (recommend,)   (spec_prior, 

In [24]:
worst_combination = result_df[result_df['AUC']==result_df['AUC'].min()]
worst_combination['Negative']

5    (spec_prior, very_recom, not_recom)
Name: Negative, dtype: object

In [25]:
result_df

,Positive,Negative,AUC
0,"(recommend,)","(spec_prior, very_recom, priority, not_recom)",1.000000
1,"(priority,)","(spec_prior, recommend, very_recom, not_recom)",0.999875
2,"(not_recom,)","(spec_prior, recommend, very_recom, priority)",1.000000
3,"(very_recom,)","(spec_prior, recommend, priority, not_recom)",0.999903
4,"(spec_prior,)","(recommend, very_recom, priority, not_recom)",0.999910
5,"(recommend, priority)","(spec_prior, very_recom, not_recom)",0.999772
6,"(recommend, not_recom)","(spec_prior, very_recom, priority)",1.000000
7,"(recommend, very_recom)","(spec_prior, priority, not_recom)",0.999995
8,"(recommend, spec_prior)","(very_recom, priority, not_recom)",0.999876
9,"(priority, not_recom)","(spec_prior, recommend, very_recom)",0.999862


In [26]:
salve = generate_combinations(worst_combination['Negative'].iloc[0], half=True)
salve

[(('spec_prior',), ('very_recom', 'not_recom')),
 (('very_recom',), ('spec_prior', 'not_recom')),
 (('not_recom',), ('spec_prior', 'very_recom'))]

In [27]:
salve_1d = [item for sublist in salve for item in sublist]
salve_1d[0]

('spec_prior',)

In [28]:
df2 = df.copy()
df2.loc[df2['class'].isin(['recommend', 'priority']), 'class'] = 'P'
df2 = df2[~df2['class'].isin(['not_recom', 'very_recom'])]
df2

,parents,has_nurs,form,children,housing,finance,social,health,class
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,P
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,P
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,P
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,P
6,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,P
...,...,...,...,...,...,...,...,...,...
12952,2.0,4.0,3.0,3.0,2.0,1.0,0.0,1.0,spec_prior
12954,2.0,4.0,3.0,3.0,2.0,1.0,1.0,0.0,spec_prior
12955,2.0,4.0,3.0,3.0,2.0,1.0,1.0,1.0,spec_prior
12957,2.0,4.0,3.0,3.0,2.0,1.0,2.0,0.0,spec_prior


In [29]:
result_final_df = pd.DataFrame(columns=['Positive', 'Negative', 'AUC'])
positive = worst_combination['Positive'].iloc[0]

df2 = df.copy()
df2.loc[df2['class'].isin(['recommend', 'priority']), 'class'] = 'P'
df2 = df2[~df2['class'].isin(['not_recom', 'very_recom'])]

for group in salve_1d:

    print(positive)
    print(group)

    # df_copy = df.copy()
    # df_copy['class'] = df_copy['class'].apply(lambda x: 'N' if x in group else x)

    # auc = generate_prediction(df_copy)

    # result = {'Positive': group1, 'Negative': group2, 'AUC': auc}
    # result_final_df = pd.concat([result_final_df, pd.DataFrame([result])], ignore_index=True)
    # print(result_final_df)

('recommend', 'priority')
('spec_prior',)
('recommend', 'priority')
('very_recom', 'not_recom')
('recommend', 'priority')
('very_recom',)
('recommend', 'priority')
('spec_prior', 'not_recom')
('recommend', 'priority')
('not_recom',)
('recommend', 'priority')
('spec_prior', 'very_recom')


## 2nd strategy (random 50 repetitions)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pdb

def generate_prediction(df):
    # Split the data into features and target
    X = df.drop('class', axis=1)
    y = df['class']

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=40)

    # Create and train the Random Forest model
    rf_model = LogisticRegression(random_state=40)
    rf_model.fit(X_train, y_train)

    # Make predictions with the Random Forest model
    rf_y_pred = rf_model.predict(X_test)
    # pdb.set_trace()

    # Evaluate the Random Forest model using AUC metric
    rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1])
    return rf_auc

In [ ]:
import random, pdb
from tqdm import tqdm

# random.seed(45)
search_df = pd.DataFrame(columns=['Positive', 'Negative', 'Easy', 'Hard'])

df = df[df['class'] != 'recommend']

n_classes = len(df['class'].unique())

for i in tqdm(range(50), desc="Total Classes"):

    classes_size = random.randint(3, n_classes)

    for j in tqdm(range(50), desc="Positive and Negative", leave=False):
        pos_size = random.randint(1, classes_size-2)
        pos_class = random.sample(list(df['class'].unique()), k=pos_size)

        neg_size = classes_size - pos_size
        neg_class = [x for x in df['class'].unique() if x not in pos_class]
        neg_class = random.sample(neg_class, k=neg_size)
        # print(classes_size, pos_class, neg_class)

        df_pos_neg = df.copy()
        df_pos_neg = df_pos_neg[df_pos_neg['class'].isin(pos_class + neg_class)]
        df_pos_neg['class'] = df_pos_neg['class'].apply(lambda x: 'P' if x in pos_class else 'N')

        for k in tqdm(range(50), desc="Easy and Hard", leave=False):
            hard_size = random.randint(1, neg_size-1)
            hard_class = random.sample(neg_class, k=hard_size)

            easy_size = neg_size - hard_size
            easy_class = [x for x in neg_class if x not in hard_class]
            easy_class = random.sample(easy_class, k=easy_size)

            df_pos_easy = df.copy()
            df_pos_easy = df_pos_easy[df_pos_easy['class'].isin(pos_class + easy_class)]
            df_pos_easy['class'] = df_pos_easy['class'].apply(lambda x: 'P' if x in pos_class else 'N')

            df_pos_hard = df.copy()
            df_pos_hard = df_pos_hard[df_pos_hard['class'].isin(pos_class + hard_class)]
            df_pos_hard['class'] = df_pos_hard['class'].apply(lambda x: 'P' if x in pos_class else 'N')

            result = {'Positive': pos_class, 'Negative': neg_class, 
                      'Easy': easy_class, 'Hard': hard_class}
            search_df = pd.concat([search_df, pd.DataFrame([result])], ignore_index=True)
            # print(search_df)

In [31]:
import random, pdb

# random.seed(45)
search_df = pd.DataFrame(columns=['Positive', 'Negative', 'Easy', 'Hard', 'AUC', 'AUC_E', 'AUC_H'])

df = df[df['class'] != 'recommend']

n_classes = len(df['class'].unique())

for i in range(50):

    classes_size = random.randint(3, n_classes)

    for j in range(50):
        pos_size = random.randint(1, classes_size-2)
        pos_class = random.sample(list(df['class'].unique()), k=pos_size)

        neg_size = classes_size - pos_size
        neg_class = [x for x in df['class'].unique() if x not in pos_class]
        neg_class = random.sample(neg_class, k=neg_size)
        print(classes_size, pos_class, neg_class)

        df_pos_neg = df.copy()
        df_pos_neg = df_pos_neg[df_pos_neg['class'].isin(pos_class + neg_class)]
        df_pos_neg['class'] = df_pos_neg['class'].apply(lambda x: 'P' if x in pos_class else 'N')

        # Ensure both classes are present
        if len(df_pos_neg['class'].unique()) < 2:
            print("Skipping iteration: Only one class present in y_true")
            continue

        auc = generate_prediction(df_pos_neg)
        # pdb.set_trace()

        for k in range(50):
            hard_size = random.randint(1, neg_size-1)
            hard_class = random.sample(neg_class, k=hard_size)

            easy_size = neg_size - hard_size
            easy_class = [x for x in neg_class if x not in hard_class]
            easy_class = random.sample(easy_class, k=easy_size)

            df_pos_easy = df.copy()
            df_pos_easy = df_pos_easy[df_pos_easy['class'].isin(pos_class + easy_class)]
            df_pos_easy['class'] = df_pos_easy['class'].apply(lambda x: 'P' if x in pos_class else 'N')
            auc_easy = generate_prediction(df_pos_easy)

            df_pos_hard = df.copy()
            df_pos_hard = df_pos_hard[df_pos_hard['class'].isin(pos_class + hard_class)]
            df_pos_hard['class'] = df_pos_hard['class'].apply(lambda x: 'P' if x in pos_class else 'N')
            auc_hard = generate_prediction(df_pos_hard)

            result = {'Positive': pos_class, 'Negative': neg_class, 
                      'Easy':easy_class, 'Hard': hard_class,
                      'AUC': auc, 'AUC_E': auc_easy, 'AUC_H': auc_hard}
            search_df = pd.concat([search_df, pd.DataFrame([result])], ignore_index=True)
            print(search_df)

3 ['priority'] ['not_recom', 'spec_prior']
     Positive                 Negative          Easy         Hard       AUC  \
0  [priority]  [not_recom, spec_prior]  [spec_prior]  [not_recom]  0.958485   

      AUC_E  AUC_H  
0  0.950148    1.0  


C:\Users\joaop\AppData\Local\Temp\ipykernel_3576\4203937564.py:56: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



     Positive                 Negative          Easy         Hard       AUC  \
0  [priority]  [not_recom, spec_prior]  [spec_prior]  [not_recom]  0.958485   
1  [priority]  [not_recom, spec_prior]  [spec_prior]  [not_recom]  0.958485   

      AUC_E  AUC_H  
0  0.950148    1.0  
1  0.950148    1.0  
     Positive                 Negative          Easy          Hard       AUC  \
0  [priority]  [not_recom, spec_prior]  [spec_prior]   [not_recom]  0.958485   
1  [priority]  [not_recom, spec_prior]  [spec_prior]   [not_recom]  0.958485   
2  [priority]  [not_recom, spec_prior]   [not_recom]  [spec_prior]  0.958485   

      AUC_E     AUC_H  
0  0.950148  1.000000  
1  0.950148  1.000000  
2  1.000000  0.950148  
     Positive                 Negative          Easy          Hard       AUC  \
0  [priority]  [not_recom, spec_prior]  [spec_prior]   [not_recom]  0.958485   
1  [priority]  [not_recom, spec_prior]  [spec_prior]   [not_recom]  0.958485   
2  [priority]  [not_recom, spec_prior]   [

KeyboardInterrupt: 

In [89]:
search_df

,Positive,Negative,Hard,AUC,AUC_H
0,[priority],"[spec_prior, not_recom, very_recom, recommend]",[not_recom],0.999875,1.000000
1,[priority],"[spec_prior, not_recom, very_recom, recommend]","[spec_prior, recommend, very_recom]",0.999875,0.999824
2,[priority],"[spec_prior, not_recom, very_recom, recommend]","[recommend, spec_prior, very_recom]",0.999875,0.999824
3,[priority],"[spec_prior, not_recom, very_recom, recommend]",[not_recom],0.999875,1.000000
4,[priority],"[spec_prior, not_recom, very_recom, recommend]","[very_recom, not_recom, spec_prior]",0.999875,0.999833
...,...,...,...,...,...
282,[recommend],"[not_recom, spec_prior, priority, very_recom]","[not_recom, spec_prior, priority, very_recom]",1.000000,1.000000
283,[recommend],"[not_recom, spec_prior, priority, very_recom]","[priority, very_recom]",1.000000,0.998351
284,[recommend],"[not_recom, spec_prior, priority, very_recom]",[very_recom],1.000000,NaN
285,[recommend],"[not_recom, spec_prior, priority, very_recom]","[not_recom, priority, spec_prior, very_recom]",1.000000,1.000000


In [90]:
df[df['class']=='recommend']

,parents,has_nurs,form,children,housing,finance,social,health,class
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,recommend
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,recommend
